In [ ]:
import logging
import os
import re
import time

import nest_asyncio

from gmas.builder import GraphBuilder
from gmas.execution import MACPRunner, RunnerConfig, StreamEventType
from gmas.tools import WebSearchTool, create_openai_caller, register_tool

os.environ.setdefault("NO_PROXY", "*")  # bypass Windows system proxy
logging.disable(logging.INFO)  # silence noisy HTTP / framework logs
nest_asyncio.apply()

# ── Shared LLM caller ────────────────────────────────────────────
caller = create_openai_caller(
    base_url="",
    api_key="",
    model="",
    temperature=0,
    max_tokens=4_000,
    tool_choice="auto",
)


# ── Helper: run query and print answer + visited URLs ─────────────────
def _run(graph, runner, query):
    graph.update_task(query)
    urls = []
    t0 = time.perf_counter()
    for ev in runner.stream(graph):
        if ev.event_type == StreamEventType.AGENT_OUTPUT and ev.content:
            urls += re.findall(r'https?://[^\s\]"<>)]+', ev.content)
        elif ev.event_type == StreamEventType.RUN_END:
            wall = time.perf_counter() - t0
            print(ev.final_answer)
            urls += re.findall(r'https?://[^\s\]"<>)]+', ev.final_answer or "")
            seen = dict.fromkeys(urls)
            if seen:
                print(f"\n🌐 Sources ({len(seen)}):")
                for i, u in enumerate(seen, 1):
                    print(f"   {i}. {u}")
            print(f"\n⏱ {wall:.1f}s wall | {ev.total_time:.1f}s runner | tokens: {ev.total_tokens}")

In [2]:
# ── Web Search agent ──────────────────────────────────────────────────────────
register_tool(
    WebSearchTool(
        max_results=10,
        max_content_length=12000,
        fetch_content=True,
        max_fetch_pages=10,
        timeout=15,
    )
)

PERSONA = """
You are a fast web research agent. Answer the user's question concisely.
"""
DESCRIPTION = """
perform an analysis to answer the user's question using web search for information,
do not invent anything, use only information from the Internet, and search until you find what you need.
REQUIRED: additionally, write down the relevant links that you visited for the search.
"""
b = GraphBuilder()
b.add_agent("researcher", persona=PERSONA, description=DESCRIPTION, tools=["web_search"])
b.add_task(query="(placeholder)")
b.connect_task_to_agents(agent_ids=["researcher"])
graph = b.build()
runner = MACPRunner(llm_caller=caller, config=RunnerConfig(max_tool_iterations=2, timeout=60))


def run_query(q):
    _run(graph, runner, q)


print("✅ web_search ready")

✅ web_search ready


In [3]:
run_query(
    "что из последнего по статьям агентных систем вышло, за последнюю неделю (16 марта 2026 года),"
    " только актуальные, не старые, хотяб за неделю от сегодняшнего числа не просрочены, в первую очередь ищи на архиве"
)

**Новые статьи по агентным (мульти‑агентным) системам за последнюю неделю (≈ 16 – 23 марта 2026 г.)**  
*Источник — архив arXiv, раздел **cs.MA – Multiagent Systems** (https://arxiv.org/list/cs.MA/recent). Все перечисленные работы появились в arXiv в период с 16 по 20 марта 2026 г.; они находятся в “fresh”‑листинге и ещё не прошли рецензирование в журналах, поэтому полностью актуальны.*

| № | arXiv‑идентификатор | Дата публикации* | Заголовок | Авторы | Краткое содержание (1‑2 предложения) | Ссылка |
|---|--------------------|------------------|-----------|--------|--------------------------------------|--------|
| 1 | **2603.18096** | 20 марта 2026 | *A Trace‑Based Assurance Framework for Agentic AI Orchestration: Contracts, Testing, and Governance* | Ciprian Paduraru, Petru‑Liviu Bouruc, Alin Stefanescu | Предлагает формальный фреймворк трассировки для проверки и управления оркестрацией агентных ИИ‑систем, включая контракты и тест‑планы. | https://arxiv.org/abs/2603.18096 |
| 2 | **

In [4]:
# ── Deep Search agent (web_search + Selenium) ────────────────────────────────
from gmas.tools import ToolRegistry

deep_registry = ToolRegistry()
_deep_tool = WebSearchTool(
    max_results=10,
    max_content_length=12000,
    fetch_content=True,
    max_fetch_pages=5,
    timeout=25,
    deep_search="selenium",
    browser_config={
        "headless": True,
        "browser": "auto",
        "scroll_to_bottom": True,
        "max_scrolls": 3,
        "disable_images": True,
    },
)
_deep_tool.warm_up()
deep_registry.register(_deep_tool)

DEEP_PERSONA = """
You are a fast web research agent. Answer the user's question concisely.
"""
DEEP_DESCRIPTION = """
проведи анализ для ответа на вопрос пользователя, используя web search, сам не придумывай информацию.
ОБЯЗАТЕЛЬНО: дополнительно напиши актуальные ссылки, на которые ты заходил для поиска.
"""

b2 = GraphBuilder()
b2.add_agent("deep_researcher", persona=DEEP_PERSONA, description=DEEP_DESCRIPTION, tools=["web_search"])
b2.add_task(query="(placeholder)")
b2.connect_task_to_agents(agent_ids=["deep_researcher"])
deep_graph = b2.build()
deep_runner = MACPRunner(
    llm_caller=caller,
    config=RunnerConfig(max_tool_iterations=4, timeout=180, tool_registry=deep_registry),
)


def run_deep_query(q):
    _run(deep_graph, deep_runner, q)


print("✅ deep_search (Selenium) ready")

✅ deep_search (Selenium) ready


In [5]:
run_deep_query(
    "что из последнего по статьям агентных систем вышло, за последнюю неделю (16 марта 2026),"
    " только актуальные, не старые, хотяб за неделю от сегодняшнего числа не просрочены, в первую очередь ищи на архиве"
)

**Последняя неделя (≈ 16 – 23 марта 2026) — самые свежие публикации о агентных (agentic) системах**  

| № | Дата публикации | Заголовок (кратко) | Где опубликовано | Краткое содержание | Ссылка |
|---|----------------|--------------------|------------------|--------------------|--------|
| 1 | **15 марта 2026** (еженедельный дайджест) | **“The LLM and AI Agent Releases That Actually Matter This Week, March 2026”** | Dev.to (технический блог) | Обзор самых значимых релизов больших языковых моделей и автономных AI‑агентов за неделю: GPT‑5.4 от OpenAI (контекст 1 млн токенов), Anthropic Claude 3.2‑Pro, Alibaba Qwen 3.5 9B, а также новые фреймворки для построения multi‑agent систем (LangChain 2.0, Auto‑GPT‑Pro). В статье указаны ссылки на репозитории, основные метрики и примеры применения в бизнес‑процессах. | https://dev.to/aibughunter/the-llm-and-ai-agent-releases-that-actually-matter-this-week-march-2026-5d7i |
| 2 | **17 марта 2026** | **“AI Agents 2026: Agentic Revolution – OnDemand 

In [6]:
# ── Deep Search agent (web_search + Playwright) ──────────────────────────────
from gmas.tools import ToolRegistry

pw_registry = ToolRegistry()
_pw_tool = WebSearchTool(
    max_results=10,
    max_content_length=12000,
    fetch_content=True,
    max_fetch_pages=5,
    timeout=25,
    deep_search="playwright",
    browser_config={
        "headless": True,
        "browser": "chromium",
        "scroll_to_bottom": True,
        "max_scrolls": 3,
        "disable_images": True,
    },
)
_pw_tool.warm_up()
pw_registry.register(_pw_tool)

PW_PERSONA = """
You are a fast web research agent. Answer the user's question concisely.
"""
PW_DESCRIPTION = """
проведи анализ для ответа на вопрос пользователя используя обязетельно web search, сам не придумывай информацию,
ОБЯЗАТЕЛЬНО: дополнительно напиши актуальные ссылки, на которые ты заходил для поиска. Если не заходил — не пиши.
"""

b3 = GraphBuilder()
b3.add_agent("pw_researcher", persona=PW_PERSONA, description=PW_DESCRIPTION, tools=["web_search"])
b3.add_task(query="(placeholder)")
b3.connect_task_to_agents(agent_ids=["pw_researcher"])
pw_graph = b3.build()
pw_runner = MACPRunner(
    llm_caller=caller,
    config=RunnerConfig(max_tool_iterations=4, timeout=180, tool_registry=pw_registry),
)


def run_pw_query(q):
    _run(pw_graph, pw_runner, q)


print("✅ deep_search (Playwright) ready")

✅ deep_search (Playwright) ready


In [7]:
run_pw_query(
    "что из последнего по статьям агентных систем вышло, за последнюю неделю (16 марта 2026),"
    " только актуальные, не старые, хотяб за неделю от сегодня не просрочены, в первую очередь ищи на архиве"
)

Provider DuckDuckGoProvider failed for 'agent systems March 2026 site:arxiv.org' (api_error, status=None) — trying next
All providers exhausted for 'agent systems March 2026 site:arxiv.org'; 1 failed, 0 returned no results


**Последняя неделя (9 – 16 марта 2026) — новейшие публикации о агентных (agentic) системах**  
(все источники получены через поиск в открытых репозиториях / интернет‑архива; ссылки активны на момент 23 марта 2026).

| № | Название (англ.) | Дата публикации (arXiv) | Краткое содержание | Ссылка |
|---|-------------------|--------------------------|--------------------|--------|
| 1 | **AgentRM: An OS‑Inspired Resource Manager for LLM Agent Systems** | 13 марта 2026 (v1) | Предлагает middleware‑уровень, аналогичный операционной системе, для управления ресурсами LLM‑агентов (планировщик MLFQ, контроль лимитов, менеджер контекста). Показано снижение 95‑го перцентиля задержки на 86 % и рост пропускной способности на 168 % по сравнению с базовыми решениями. | https://arxiv.org/abs/2603.13110 |
| 2 | **Behind the Prompt: The Agent‑User Problem in Information Retrieval** | 9 марта 2026 (v1) | Анализирует, как традиционные модели пользовательского поведения в IR‑системах ломаются, когда «пользо

In [8]:
run_pw_query("какое сегодня число и сколько времени, проверь также какая у меня скорость интернета")

**Сегодняшняя дата:** 23 марта 2026 г.  
**Текущее время (UTC):** 00 : 33 : 35 (по данным WorldTimeAPI)【http://worldtimeapi.org/api/timezone/Etc/UTC】.  

**Скорость вашего интернета:** Я не могу измерить или проверить вашу текущую скорость соединения, так как это требует выполнения теста на вашем устройстве (например, через speedtest.net).  

**Источники**  
- Календарные сайты, подтверждающие, что 23 марта 2026 г. — [Wincalendar](https://www.wincalendar.com/Calendar/Date/March-23-2026), [Date.Tools](https://date.tools/2026-03-23).  
- Текущее время в UTC — [WorldTimeAPI](http://worldtimeapi.org/api/timezone/Etc/UTC).  

🌐 Sources (4):
   1. http://worldtimeapi.org/api/timezone/Etc/UTC】.
   2. https://www.wincalendar.com/Calendar/Date/March-23-2026
   3. https://date.tools/2026-03-23
   4. http://worldtimeapi.org/api/timezone/Etc/UTC

⏱ 17.5s wall | 17.5s runner | tokens: 1672
